## Data Modelling

In [26]:
import os # get working directory
import pandas as pd
import numpy as np
import re # regular expression
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split

### Pre-modelling assessment

In [22]:
hospital_los_train_final = pd.read_csv("/workspaces/myfolder/project/sasviya_challenge_2026/data/hospital_los_train_final.csv")
hospital_los_train_final.head(10)

,ICU_DAYS,NUM_CHRONIC_COND,ORDER_SET_USED,ORDER_TOTAL_CHARGES,X,Y,PATIENT_AGE,ICD9_TARGET,DRG_APR_SEVERITY,OPERATION_COUNT,...,icu_x_severity,visits_x_chronic,icu_x_comorbidity,careteam_x_comorbidity,comorbidity_x_age,age_group_40_54,age_group_55_64,age_group_65_74,age_group_75_84,age_group_85_plus
0,4,2,0,13691,-77.18702,38.773578,64,1,3,0,...,12,2,8,6,128,False,True,False,False,False
1,1,0,1,33790,-77.18702,38.773578,98,1,1,1,...,1,0,0,0,0,False,False,False,False,True
2,0,1,1,17601,-77.18702,38.773578,75,1,2,2,...,0,0,0,25,375,False,False,True,False,False
3,0,1,1,11565,-77.18702,38.773578,81,1,2,0,...,0,4,0,0,0,False,False,False,True,False
4,6,1,1,43357,-77.18702,38.773578,81,0,2,1,...,12,2,12,2,162,False,False,False,True,False
5,7,2,1,37106,-77.18702,38.773578,69,1,3,1,...,21,2,56,40,552,False,False,True,False,False
6,0,1,1,13812,-77.18702,38.773578,87,1,4,1,...,0,2,0,9,261,False,False,False,False,True
7,3,0,1,14853,-77.18702,38.773578,77,1,2,1,...,6,0,9,6,231,False,False,False,True,False
8,3,1,1,27145,-77.18702,38.773578,85,1,3,1,...,9,0,9,9,255,False,False,False,True,False
9,3,1,1,13416,-77.18702,38.773578,76,1,2,0,...,6,2,0,0,0,False,False,False,True,False


In [23]:
hospital_los_test_final = pd.read_csv("/workspaces/myfolder/project/sasviya_challenge_2026/data/hospital_los_test_final.csv")
hospital_los_test_final.head(10)

,ICU_DAYS,NUM_CHRONIC_COND,ORDER_SET_USED,ORDER_TOTAL_CHARGES,X,Y,PATIENT_AGE,ICD9_TARGET,DRG_APR_SEVERITY,OPERATION_COUNT,...,icu_x_severity,visits_x_chronic,icu_x_comorbidity,careteam_x_comorbidity,comorbidity_x_age,age_group_40_54,age_group_55_64,age_group_65_74,age_group_75_84,age_group_85_plus
0,2,2,0,200,-77.18702,38.773578,64,0,2,0,...,4,30,4,6,128,False,True,False,False,False
1,1,1,1,9027,-77.18702,38.773578,67,1,2,0,...,2,2,0,0,0,False,False,True,False,False
2,3,1,1,10885,-77.18702,38.773578,73,0,2,0,...,6,9,3,1,73,False,False,True,False,False
3,1,1,1,8289,-77.18702,38.773578,92,0,1,1,...,1,1,2,4,184,False,False,False,False,True
4,2,1,1,26815,-77.18702,38.773578,68,1,3,0,...,6,3,0,0,0,False,False,True,False,False
5,0,1,1,16214,-77.18702,38.773578,68,1,4,0,...,0,2,0,20,340,False,False,True,False,False
6,0,2,0,16039,-77.18702,38.773578,76,1,3,1,...,0,0,0,16,304,False,False,False,True,False
7,6,0,1,28490,-77.18702,38.773578,71,1,2,1,...,12,0,18,12,213,False,False,True,False,False
8,2,1,1,21391,-77.18702,38.773578,82,0,3,1,...,6,1,8,16,328,False,False,False,True,False
9,6,0,1,31061,-77.18702,38.773578,77,1,3,0,...,18,0,12,10,154,False,False,False,True,False


In [ ]:
# confirm no missing values and the breakdown of data types
print(hospital_los_train_final.dtypes.value_counts())
print(hospital_los_train_final.isnull().sum().sum(), hospital_los_test_final.isnull().sum().sum())  # should both be 0

bool       48
int64      24
float64    10
Name: count, dtype: int64
0 0


In [5]:
corr_matrix = hospital_los_train_final.drop(columns=['ADMIT_LOS']).corr()
high_corr = corr_matrix.where((corr_matrix.abs() > 0.85) & (corr_matrix.abs() < 1.0))
high_corr_pairs = high_corr.stack().reset_index()
high_corr_pairs.columns = ['feature_1', 'feature_2', 'correlation']
print(high_corr_pairs.sort_values('correlation', key=abs, ascending=False))

                           feature_1                         feature_2  \
6                  COMORBIDITY_INDEX                 comorbidity_x_age   
21                 comorbidity_x_age                 COMORBIDITY_INDEX   
15                    icu_x_severity                          ICU_DAYS   
0                           ICU_DAYS                    icu_x_severity   
4                  COMORBIDITY_INDEX                    chronic_burden   
7                     chronic_burden                 COMORBIDITY_INDEX   
8                     chronic_burden                 comorbidity_x_age   
22                 comorbidity_x_age                    chronic_burden   
19            careteam_x_comorbidity                 COMORBIDITY_INDEX   
5                  COMORBIDITY_INDEX            careteam_x_comorbidity   
18                 icu_x_comorbidity                    icu_x_severity   
16                    icu_x_severity                 icu_x_comorbidity   
10                  dist_to_hospital  

In [13]:
# Drop chronic_burden - confirmed exact linear combination of NUM_CHRONIC_COND + COMORBIDITY_INDEX,
# which caused perfect multicollinearity (R²=1.0, divide-by-zero in VIF)
X = hospital_los_train_final.drop(columns=['ADMIT_LOS', 'chronic_burden'])
X = X.select_dtypes(include=['number', 'bool'])
X = X.astype({col: 'int64' for col in X.select_dtypes(include='bool').columns})

X_sample = X.sample(n=10000, random_state=42)
vif_data = pd.DataFrame()
vif_data['feature'] = X_sample.columns
vif_data['VIF'] = [variance_inflation_factor(X_sample.values, i) for i in range(X_sample.shape[1])]
print(vif_data.sort_values('VIF', ascending=False).head(15))

                              feature          VIF
4                                   X  6039.909109
67                HOSPITAL_target_enc  3534.265310
5                                   Y  1055.270404
6                         PATIENT_AGE   552.484377
69   DIAGNOSIS_SUBCAT_CODE_target_enc   437.782866
68                DX_GROUP_target_enc   410.601958
18      DIAGNOSIS_ICD_CODE_target_enc   162.235444
59                       STATECODE_FL   109.712167
11                  COMORBIDITY_INDEX    81.090266
21            DRG_APR_CODE_target_enc    77.867611
78                    age_group_75_84    72.409681
46  DISCHARGED_TO_ROUTINE DSCHG, HOME    66.864078
74                  comorbidity_x_age    66.313457
79                  age_group_85_plus    63.580095
20             MS_DRG_CODE_target_enc    56.098942


In [ ]:
# create a log-transformed column for ADMIT_LOS
hospital_los_train_final['ADMIT_LOS_log'] = np.log1p(hospital_los_train_final['ADMIT_LOS'])

In [25]:
# Final feature trim based on correlation/VIF findings:
# drop features that are largely redundant with another feature already kept in the set

# chronic_burden is an exact linear combination of NUM_CHRONIC_COND + COMORBIDITY_INDEX (caused VIF divide-by-zero)
# MS_DRG_CODE/DRG_APR_CODE/DX_GROUP/DIAGNOSIS_SUBCAT_CODE target encodings overlap heavily with DIAGNOSIS_ICD_CODE (same diagnostic hierarchy)
# STATECODE/REGION dummies and dist_to_hospital overlap heavily with X/Y coordinates (same geography signal)
# age_group dummies are a coarser restatement of PATIENT_AGE, already in the feature set

cols_to_drop = [
    'chronic_burden',
    'MS_DRG_CODE_target_enc', 'DRG_APR_CODE_target_enc', 
    'DX_GROUP_target_enc', 'DIAGNOSIS_SUBCAT_CODE_target_enc',
    'STATECODE_AR', 'STATECODE_FL', 'STATECODE_GA', 'STATECODE_IL', 
    'STATECODE_MO', 'STATECODE_MS', 'STATECODE_TN', 'STATECODE_TX', 'STATECODE_VA',
    'REGION_Region 2', 'REGION_Region 3', 'REGION_Region 4', 'REGION_Region 5',
    'REGION_Region 6', 'REGION_Region 7', 'REGION_Region 8', 'REGION_Region 9',
    'REGION_Region 10', 'REGION_Region 11',
    'dist_to_hospital',
    'age_group_40_54', 'age_group_55_64', 'age_group_65_74', 'age_group_75_84', 'age_group_85_plus'
]

hospital_los_train_final = hospital_los_train_final.drop(columns=cols_to_drop)
hospital_los_test_final = hospital_los_test_final.drop(columns=[c for c in cols_to_drop if c in hospital_los_test_final.columns])

print(hospital_los_train_final.shape, hospital_los_test_final.shape)
print(hospital_los_train_final.columns.tolist())
print(hospital_los_test_final.columns.tolist())

(100000, 53) (15000, 51)
['ICU_DAYS', 'NUM_CHRONIC_COND', 'ORDER_SET_USED', 'ORDER_TOTAL_CHARGES', 'X', 'Y', 'PATIENT_AGE', 'ICD9_TARGET', 'DRG_APR_SEVERITY', 'OPERATION_COUNT', 'MONITORING_HOURS', 'COMORBIDITY_INDEX', 'CARE_TEAM_SIZE', 'ADMIT_MTH', 'NUM_VISITS', 'ADMIT_LOS', 'had_procedure', 'STANDARD_ORDERS_USED', 'high_severity_flag', 'DIAGNOSIS_ICD_CODE_target_enc', 'PROCEDURE_ICD_CODE_target_enc', 'geo_cluster', 'DEPARTMENT_GENERAL SURG', 'DEPARTMENT_HEART', 'DEPARTMENT_NEUROSCIENCES', 'DEPARTMENT_ONCOLOGY', 'DEPARTMENT_PSYCH', 'DEPARTMENT_PULMONARY', 'DEPARTMENT_TRANSPLANT', 'DEPARTMENT_Unknown', 'DEPARTMENT_WOMENS', 'GENDER_M', 'RACE_CD_Others', 'RACE_CD_White', 'DIAGNOSIS_GROUP_CHF', 'DIAGNOSIS_GROUP_COPD', 'DISCHARGED_TO_CHG TO LTAC', 'DISCHARGED_TO_HOME HEALTH AGENCY', 'DISCHARGED_TO_HOSPICE (HOME)', 'DISCHARGED_TO_HOSPICE - MEDICAL INP', 'DISCHARGED_TO_INTERMEDIATE CARE', 'DISCHARGED_TO_OTHER ACUTE HOSP', 'DISCHARGED_TO_OTHER DEATH', 'DISCHARGED_TO_REHAB HOSPITAL', 'DISCHARG

In [27]:
X = hospital_los_train_final.drop(columns=['ADMIT_LOS', 'ADMIT_LOS_log'])
y = hospital_los_train_final['ADMIT_LOS_log']  # using log-transformed target given the 3.26 skew

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_val.shape)

(80000, 51) (20000, 51)
